# Primeri korišćenja COMtext.RS modela

U datim primerima, koristiće se modeli za ekavicu. Isti kod bi se primenjivao i za ijekavicu, sa izmenom modela i par promenljivih u kodu.

Pretpostavka je da su delovi koda iz ovog repozitorijuma korišćeni, primarno klasa za lematizaciju (src\helpers\lemmatizer).

In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))
print(ROOT_DIR)

d:\Work\ICEF\comtext-work


## Rad sa slobodnim tekstom

Pod pretpostavkom da je polazna tačka slobodan tekst, prva stvar koju treba uraditi je tokenizacija. U tu svrhu se koristi CLASSLA-Stanza tokenizator.

In [3]:
import classla 
classla.download('sr', processors = 'tokenize')
tokenizer = classla.Pipeline('sr', processors = 'tokenize')

2026-11-17 01:08:38 INFO: Downloading these customized packages for language: sr (Serbian)...
| Processor | Package  |
------------------------
| tokenize  | standard |

2026-11-17 01:08:38 INFO: Finished downloading models and saved to C:\Users\Aleks\classla_resources.
2026-11-17 01:08:38 INFO: Loading these models for language: sr (Serbian):
| Processor | Package  |
------------------------
| tokenize  | standard |

2026-11-17 01:08:38 INFO: Use device: gpu
2026-11-17 01:08:38 INFO: Loading: tokenize
2026-11-17 01:08:38 INFO: Done loading processors!


# Tokenizacija i primeri rečenica

In [4]:
examples = [
    'Sudija je izdao propis 1. klase i 2. reda.', 
    'Sud je po prijemu tužbe započeo pripreme za glavnu raspravu koje su obuhvatale i dostavu tužbe tuženom na odgovor u skladu odredbi člana 274 Zakona o parničnom postupku.',
    'OBAVEZUJE SE tuženi Zorić Andrija iz Beograda ulica Sarajevska br. 101 da tužilji Kostić Mari iz Beograda ul. Miše Vujića br. 2 vrati iznos od 145.000 evra u roku od 15 dana od dana prijema pisanog otpravka presude pod pretnjom izvršenja.'
    ]

classla_tokenized = []
for sentence in examples:
    doc = tokenizer(sentence)
    tokenized_sentence = []
    for word in doc.iter_words():
        tokenized_sentence.append(word.text)
    classla_tokenized.append(tokenized_sentence)

for sentence in classla_tokenized:
    print(sentence)

['Sudija', 'je', 'izdao', 'propis', '1.', 'klase', 'i', '2.', 'reda', '.']
['Sud', 'je', 'po', 'prijemu', 'tužbe', 'započeo', 'pripreme', 'za', 'glavnu', 'raspravu', 'koje', 'su', 'obuhvatale', 'i', 'dostavu', 'tužbe', 'tuženom', 'na', 'odgovor', 'u', 'skladu', 'odredbi', 'člana', '274', 'Zakona', 'o', 'parničnom', 'postupku', '.']
['OBAVEZUJE', 'SE', 'tuženi', 'Zorić', 'Andrija', 'iz', 'Beograda', 'ulica', 'Sarajevska', 'br.', '101', 'da', 'tužilji', 'Kostić', 'Mari', 'iz', 'Beograda', 'ul', '.', 'Miše', 'Vujića', 'br.', '2', 'vrati', 'iznos', 'od', '145.000', 'evra', 'u', 'roku', 'od', '15', 'dana', 'od', 'dana', 'prijema', 'pisanog', 'otpravka', 'presude', 'pod', 'pretnjom', 'izvršenja', '.']


## Model za morfosintaktičko označavanje

Model za ijekavicu: ICEF-NLP/bcms-bertic-comtext-sr-legal-msd-ijekavica. U ovom slučaju je to jedina stvar koju treba izmeniti.

Bez obzira da li se koristi ovaj ili neki od sledećih modela predstavljenih ovde, treba imati dve stvari u vidu:
1. Biblioteka simpletransformers zahteva da labele budu u redosledu u kom su bile kada je model treniran ([zadatak]_label.json fajlovi u exmples direktorijumu), inače modeli izbacuju pogrešne oznake
2. Biblioteka simpletransformers kao izlaz funkcije predict ima rečnik oblika {"ulazni_token" : "oznaka_koju_je_model_dodelio"} 


In [5]:
import json
from simpletransformers.ner import NERModel

with open('msd_labels.json', 'r') as label_file:
    msd_labels = json.load(label_file)

msd_model = NERModel('electra', 'ICEF-NLP/bcms-bertic-comtext-sr-legal-msd-ekavica', use_cuda=False, labels=msd_labels)
msd_predictions_list, _ = msd_model.predict(classla_tokenized, split_on_space=False)

for pred_list in msd_predictions_list:
    for pair in pred_list:
        word, msd = next(iter(pair.items()))
        print(f"{word} - {msd}")
    print("\n")

  0%|          | 0/1 [00:00<?, ?it/s]

Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

Sudija - Ncmsn
je - Var3s
izdao - Vmp-sm
propis - Ncmsan
1. - Mdo
klase - Ncfsg
i - Cc
2. - Mdo
reda - Ncmsg
. - Z


Sud - Ncmsn
je - Var3s
po - Sl
prijemu - Ncmsl
tužbe - Ncfsg
započeo - Vmp-sm
pripreme - Ncfpa
za - Sa
glavnu - Agpfsay
raspravu - Ncfsa
koje - Pi-fpn
su - Var3p
obuhvatale - Vmp-pf
i - Cc
dostavu - Ncfsa
tužbe - Ncfsg
tuženom - Agpmsdy
na - Sa
odgovor - Ncmsan
u - Sl
skladu - Ncmsl
odredbi - Ncfpg
člana - Ncmsg
274 - Mdc
Zakona - Ncmsg
o - Sl
parničnom - Agpmsly
postupku - Ncmsl
. - Z


OBAVEZUJE - Vmr3s
SE - Px--sa
tuženi - Agpmsny
Zorić - Npmsn
Andrija - Npmsn
iz - Sg
Beograda - Npmsg
ulica - Ncfsn
Sarajevska - Agpfsny
br. - Y
101 - Mdc
da - Cs
tužilji - Ncfsd
Kostić - Npmsn
Mari - Npfsd
iz - Sg
Beograda - Npmsg
ul - Y
. - Z
Miše - Npmsg
Vujića - Npmsg
br. - Y
2 - Mdc
vrati - Vmr3s
iznos - Ncmsan
od - Sg
145.000 - Mdm
evra - Ncmpg
u - Sl
roku - Ncmsl
od - Sg
15 - Mdc
dana - Ncmpg
od - Sg
dana - Ncmsg
prijema - Ncmsg
pisanog - Agpmsgy
otpravka - Ncmsg
presude - Ncfsg
p

## Lematizacija uz pomoć modela za morfosintaktičko označavanje i flektivnog leksikona

U slučaju ijekavice, klasa Lemmatizer moze da koristi i srpski(srLex) i hrvatski(hrLex) inflektivni leksikon. Ovo se kontroliše boolean oznakom is_ekavica, ako je True koristi se srLex, ako je False, koristi se hrLex. Takođe se pretpostavlja da su oba leksikona u data\ direktorijumu, ali ovo može biti izmenjeno u src\paths.py.

In [6]:
from src.helpers.lemmatizer import Lemmatizer

lemmatizer = Lemmatizer(is_ekavica=True, is_latin=True)

for pred_list in msd_predictions_list:
    for pair in pred_list:
        word, msd = next(iter(pair.items()))
        lemma = lemmatizer.get_lemma(word, msd)
        print(f"{word} - {lemma}")
    print('\n')

Sudija - sudija
je - biti
izdao - izdao
propis - propis
1. - 1.
klase - klasa
i - i
2. - 2.
reda - red
. - .


Sud - sud
je - biti
po - po
prijemu - prijem
tužbe - tužba
započeo - započeti
pripreme - priprema
za - za
glavnu - glavni
raspravu - rasprava
koje - koji
su - biti
obuhvatale - obuhvatati
i - i
dostavu - dostava
tužbe - tužba
tuženom - tuženom
na - na
odgovor - odgovor
u - u
skladu - sklad
odredbi - odredba
člana - član
274 - 274
Zakona - zakon
o - o
parničnom - parnični
postupku - postupak
. - .


OBAVEZUJE - obavezivati
SE - sebe
tuženi - tuženi
Zorić - Zorić
Andrija - Andrija
iz - iz
Beograda - Beograda
ulica - ulica
Sarajevska - sarajevski
br. - broj
101 - 101
da - da
tužilji - tužilja
Kostić - Kostić
Mari - Mari
iz - iz
Beograda - Beograda
ul - ul
. - .
Miše - Miše
Vujića - Vujića
br. - broj
2 - 2
vrati - vratiti
iznos - iznos
od - od
145.000 - 145.000
evra - evro
u - u
roku - rok
od - od
15 - 15
dana - dan
od - od
dana - dan
prijema - prijem
pisanog - pisani
otpravka - o

## Modeli za lematizaciju koji koriste transformacije teksta

Model za ijekavicu je: 'ICEF-NLP/bcms-bertic-comtext-sr-legal-lemma-ijekavica'. 

Kada bi se koristio model za ijekavicu, trebalo bi izmeniti i transformation_labels.json, sve tekstualne transformacije su dostupne u .conllu fajlovima u data direktorijumu.

In [7]:
import ast

with open('transformation_labels.json', 'r', encoding='utf-8') as f:
    transformation_labels = f.read()
transformation_labels = ast.literal_eval(transformation_labels)

lemma_trans_model = NERModel('electra', 'ICEF-NLP/bcms-bertic-comtext-sr-legal-lemma-ekavica', use_cuda=False, labels=transformation_labels)
transformation_predictions_list, _ = lemma_trans_model.predict(classla_tokenized, split_on_space=False)


for pred_list, msd_list in zip(transformation_predictions_list, msd_predictions_list):
    for pair, msd_pair in zip(pred_list, msd_list):
        word, msd = next(iter(msd_pair.items()))
        word, trans = next(iter(pair.items()))
        transform = trans.split('[')[1].split(']')[0].split(',')
        start = int(transform[2])
        end = len(word)  - int(transform[0])
        prefix =  transform[3].split("'")[1].strip() 
        suffix = transform[1].split("'")[1].strip()
        lemma = str(prefix + word[start:end] + suffix)
        lemma = lemma.lower()
        if msd.startswith('Np'):
            lemma = lemma.capitalize()
        print(f"{word} - {trans} - {lemma}")
    print("\n")

  0%|          | 0/1 [00:00<?, ?it/s]

Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

Sudija - [0, '', 0, ''] - sudija
je - [2, 'biti', 0, ''] - biti
izdao - [1, 'ti', 0, ''] - izdati
propis - [0, '', 0, ''] - propis
1. - [0, '', 0, ''] - 1.
klase - [1, 'a', 0, ''] - klasa
i - [0, '', 0, ''] - i
2. - [0, '', 0, ''] - 2.
reda - [1, '', 0, ''] - red
. - [0, '', 0, ''] - .


Sud - [0, '', 0, ''] - sud
je - [2, 'biti', 0, ''] - biti
po - [0, '', 0, ''] - po
prijemu - [1, '', 0, ''] - prijem
tužbe - [1, 'a', 0, ''] - tužba
započeo - [1, 'ti', 0, ''] - započeti
pripreme - [1, 'a', 0, ''] - priprema
za - [0, '', 0, ''] - za
glavnu - [2, 'an', 0, ''] - glavan
raspravu - [1, 'a', 0, ''] - rasprava
koje - [1, 'i', 0, ''] - koji
su - [2, 'biti', 0, ''] - biti
obuhvatale - [2, 'ti', 0, ''] - obuhvatati
i - [0, '', 0, ''] - i
dostavu - [1, 'a', 0, ''] - dostava
tužbe - [1, 'a', 0, ''] - tužba
tuženom - [2, '', 0, ''] - tužen
na - [0, '', 0, ''] - na
odgovor - [0, '', 0, ''] - odgovor
u - [0, '', 0, ''] - u
skladu - [1, '', 0, ''] - sklad
odredbi - [1, 'a', 0, ''] - odredba
člana - [

## Model za prepoznavanje imenovanih entiteta

Model za ijekavicu: ICEF-NLP/bcms-bertic-comtext-sr-legal-ner-ijekavica

Korišćenje modela za ijekavicu ne zahteva druge izmene u kodu.

In [8]:
with open('ner_labels.json', 'r', encoding='utf-8') as label_file:
    ner_labels = json.load(label_file)

ner_model = NERModel('electra', 'ICEF-NLP/bcms-bertic-comtext-sr-legal-ner-ekavica', use_cuda=False, labels=ner_labels)
ner_predictions_list, _ = ner_model.predict(classla_tokenized, split_on_space=False)

for pred_list in ner_predictions_list:
    for pair in pred_list:
        word, ner = next(iter(pair.items()))
        print(f"{word} - {ner}")
    print('\n')

  0%|          | 0/1 [00:00<?, ?it/s]

Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

Sudija - O
je - O
izdao - O
propis - O
1. - O
klase - O
i - O
2. - O
reda - O
. - O


Sud - O
je - O
po - O
prijemu - O
tužbe - O
započeo - O
pripreme - O
za - O
glavnu - O
raspravu - O
koje - O
su - O
obuhvatale - O
i - O
dostavu - O
tužbe - O
tuženom - O
na - O
odgovor - O
u - O
skladu - O
odredbi - O
člana - B-LAW
274 - I-LAW
Zakona - I-LAW
o - I-LAW
parničnom - I-LAW
postupku - I-LAW
. - O


OBAVEZUJE - O
SE - O
tuženi - O
Zorić - B-PER
Andrija - I-PER
iz - O
Beograda - B-ADR
ulica - I-ADR
Sarajevska - I-ADR
br. - I-ADR
101 - I-ADR
da - O
tužilji - O
Kostić - B-PER
Mari - I-PER
iz - O
Beograda - B-ADR
ul - I-ADR
. - I-ADR
Miše - I-ADR
Vujića - I-ADR
br. - I-ADR
2 - I-ADR
vrati - O
iznos - O
od - O
145.000 - B-MONEY
evra - I-MONEY
u - O
roku - O
od - O
15 - O
dana - O
od - O
dana - O
prijema - O
pisanog - O
otpravka - O
presude - O
pod - O
pretnjom - O
izvršenja - O
. - O


